# Cross-model scoring analysis — verification notebook

Reproduces the **figures and tables** in the appeal letter for *"Expanding functional
protein sequence space using high entropy generative models"*.

**What it does**
1. Loads the four final DCA models (bmDCA, eaDCA, edDCA, meDCA) and the *in vivo*
   relative-enrichment data.
2. **Validates** the energy function by reproducing the per-design energies stored in the
   released data to machine precision, and reproduces the published Table 1 functional rates.
3. Scores every experimentally tested design under **all four** models.
4. Produces **Result 1** (energy-penalty matrix + two figures) and **Result 2**
   (functional enzymes the low-entropy models cannot generate + the supplementary CSV).

**Data needed** (from the repo <https://github.com/robertonetti/entropy-sparse-DCA>):
- Parameters: `models/tested_models/{bmDCA,eaDCA,edDCA,meDCA}_params.zip` (git-lfs).
  Equivalently the uncompressed `.dat`, or the non-LFS `.dat.gz` in
  `models/others/old_models/`. The loader accepts `.zip`, `.dat`, or `.dat.gz`.
- Experiment: `data/experiment/` — the bmDCA/edDCA/eaDCA relative-enrichment CSV
  (use the batch that reproduces Table 1; the *Sep 2024* replicate does — this is checked
  automatically below) and the meDCA/HighEnt CSV.

Put all these files in one folder and set `BASE` to it in the **Configuration** cell.

**Energy convention.** The statistical energy is
$E(\mathbf{A})=-\sum_i h_i(a_i)-\sum_{i<j}J_{ij}(a_i,a_j)$ with
$P(\mathbf{A})\propto e^{-E(\mathbf{A})}$ (adabmDCA convention); **lower energy = higher
probability**. This matches the `energy()` routine used in the group's notebooks.


In [ ]:
import os, re, csv, glob, gzip, zipfile, io
import numpy as np
import matplotlib.pyplot as plt

## Configuration — edit `BASE` to point at your folder of files

In [ ]:
# Folder that contains the parameter files and the two CSVs (edit this):
BASE   = "."
OUTDIR = "."          # where figures and the CSV are written

MODELS  = ["eaDCA", "bmDCA", "edDCA", "meDCA"]                 # ordered by entropy (low -> high)
ENTROPY = {"eaDCA": 113.35, "bmDCA": 121.82, "edDCA": 136.05, "meDCA": 150.13}   # main-text Table
BINS    = ["20-25%", "40-45%", "60-65%"]                      # divergence from closest natural

# Candidate filenames for each model (first one found under BASE is used):
PARAM_CANDIDATES = {
    "bmDCA": ["bmDCA_params.zip", "Parameters_conv_denseDCA.dat", "Parameters_conv_CM-dense.dat.gz"],
    "eaDCA": ["eaDCA_params.zip", "Parameters_EAALG0.8.dat",      "Parameters_conv_CM-EAALG08.dat.gz"],
    "edDCA": ["edDCA_params.zip", "Parameters_conv_dec0.98.dat",  "Parameters_conv_CM-graph-dec0.98.dat.gz"],
    "meDCA": ["meDCA_params.zip", "Parameters_edDCA_density=0.125_high_entropy.dat"],
}
CSV_THREE_CANDIDATES = ["Sep2024_bmDCA_edDCA_eaDCA_CM_REdata.csv", "bmDCA_edDCA_eaDCA_CM_REdata.csv"]
CSV_MEDCA_CANDIDATES = ["meDCA_CM_REdata.csv"]

def _resolve(cands):
    for name in cands:
        hits = glob.glob(os.path.join(BASE, "**", name), recursive=True) or \
               ([os.path.join(BASE, name)] if os.path.exists(os.path.join(BASE, name)) else [])
        if hits:
            return hits[0]
    raise FileNotFoundError(f"None of {cands} found under {BASE!r}")

PARAM_FILES     = {m: _resolve(PARAM_CANDIDATES[m]) for m in MODELS}
THREE_MODEL_CSV = _resolve(CSV_THREE_CANDIDATES)
MEDCA_CSV       = _resolve(CSV_MEDCA_CANDIDATES)
for m in MODELS: print(f"{m:6s} <- {PARAM_FILES[m]}")
print("three-model CSV <-", THREE_MODEL_CSV)
print("meDCA CSV       <-", MEDCA_CSV)

## Alphabet, parameter parser, and energy function

In [ ]:
AA = {'-':0,'A':1,'C':2,'D':3,'E':4,'F':5,'G':6,'H':7,'I':8,'K':9,'L':10,
      'M':11,'N':12,'P':13,'Q':14,'R':15,'S':16,'T':17,'V':18,'W':19,'Y':20}  # q=21, gap=0
def seq2int(s): return np.array([AA.get(c, 0) for c in s], dtype=int)

def _tok(t):
    """Amino-acid column: some files use letters ('A','-'), some use integers ('1','0')."""
    return AA[t] if t in AA else int(float(t))

def _open_param(path):
    """Text stream for a parameter file given as .dat, .dat.gz, or .zip containing a .dat."""
    if path.endswith(".zip"):
        z = zipfile.ZipFile(path)
        name = [n for n in z.namelist() if n.endswith(".dat") and not n.startswith("__MACOSX")][0]
        return io.TextIOWrapper(z.open(name))
    if path.endswith(".gz"):
        return gzip.open(path, "rt")
    return open(path, "rt")

def parse_params(path):
    """Parse lines 'J i j a b value' and 'H i a value' -> symmetric J[L,q,L,q], h[L,q], density."""
    couplings, fields = [], []
    with _open_param(path) as fh:
        for line in fh:
            line = line.rstrip()
            if not line:
                continue
            if line[0] in "Jj":
                s = line[1:].split()
                couplings.append((int(s[0]), int(s[1]), _tok(s[2]), _tok(s[3]), float(s[4])))
            elif line[0] in "Hh":
                s = line[1:].split()
                fields.append((int(s[0]), _tok(s[1]), float(s[2])))
    L = max(f[0] for f in fields) + 1
    q = max(f[1] for f in fields) + 1
    J = np.zeros((L, q, L, q)); h = np.zeros((L, q)); nnz = 0
    for i, j, a, b, v in couplings:
        J[i, a, j, b] = v; J[j, b, i, a] = v           # store symmetric
        if v != 0: nnz += 1
    for i, a, v in fields:
        h[i, a] = v
    density = nnz / (q * q * L * (L - 1) / 2)           # fraction of non-zero couplings (element-wise)
    return J, h, density

def energies(J, h, seqs_int):
    """E = -sum_i h[i,a_i] - sum_{i<j} J[i,a_i,j,a_j]  (P ∝ exp(-E); lower E = higher prob)."""
    L = h.shape[0]; iu = np.triu_indices(L, k=1); idx = np.arange(L)
    out = np.empty(len(seqs_int))
    for n, a in enumerate(seqs_int):
        Jmat = J[idx[:, None], a[:, None], idx[None, :], a[None, :]]   # (L,L) of J[i,a_i,j,a_j]
        out[n] = -h[idx, a].sum() - Jmat[iu].sum()
    return out

## Load the four models and check parameter density

Densities should match Table 1: bmDCA ~100%, eaDCA ~2.8%, edDCA ~2.9%, meDCA ~12.5%.

In [ ]:
PARAMS, DENSITY = {}, {}
for m in MODELS:
    J, h, d = parse_params(PARAM_FILES[m]); PARAMS[m] = (J, h); DENSITY[m] = d
    print(f"{m}: L={h.shape[0]}  q={h.shape[1]}  density={100*d:6.3f}%")

## Load the experimental data and validate

Two checks: (1) our recomputed energies reproduce the energies stored in each design's `Name` field to machine precision; (2) the functional rates reproduce the published Table 1 (this also tells you which experimental batch to use).

In [ ]:
def load_csv(path):
    rows = []
    with open(path) as fh:
        for d in csv.DictReader(fh):
            g = d["Group"]; sp = g.split()
            m = re.search(r"(Model_Energy|DCAenergy)\s+(-?[\d.]+)", d["Name"])
            rows.append(dict(
                seq=d["Sequence"], name=d["Name"], group=g,
                model=sp[0], bin=(sp[1] if len(sp) > 1 else g),
                stored_E=float(m.group(2)) if m else np.nan,
                re=float(d["M9c Norm R.E."]) if d["M9c Norm R.E."] else np.nan,
                functional=(d["Null Mode 3Sigma Classification"].strip() == "Functional")))
    return rows

three = load_csv(THREE_MODEL_CSV)
medca = load_csv(MEDCA_CSV)
DESIGNS = [r for r in three if r["model"] in ("bmDCA", "eaDCA", "edDCA")] + \
          [r for r in medca if r["model"] == "meDCA"]
print("designs:", len(DESIGNS), "| sequence length L =", len(DESIGNS[0]["seq"]))

print("\nValidation 1 — recomputed vs stored own-model energy (expect corr=1, max|Δ|~1e-4):")
for m, rows in [("bmDCA", three), ("eaDCA", three), ("edDCA", three), ("meDCA", medca)]:
    sub = [r for r in rows if r["model"] == m and not np.isnan(r["stored_E"])]
    E = energies(*PARAMS[m], [seq2int(r["seq"]) for r in sub])
    st = np.array([r["stored_E"] for r in sub])
    print(f"  {m}: n={len(sub):4d}  corr={np.corrcoef(E, st)[0,1]:.6f}  max|Δ|={np.max(np.abs(E-st)):.2e}")

print("\nValidation 2 — functional rates (Null-mode 3σ). Compare to paper Table 1:")
print("  Table 1:  eaDCA 31.8/12.6/2.0  bmDCA 28.6/11.9/1.3  edDCA 23.9/5.2/0.0  meDCA 30.9/5.2/0.4")
for m in MODELS:
    line = f"  {m}: "
    for b in BINS:
        sub = [r for r in DESIGNS if r["model"] == m and r["bin"] == b]
        f = sum(r["functional"] for r in sub)
        line += f"{b}={100*f/len(sub):4.1f}% ({f}/{len(sub)})  "
    print(line)

## Score every design under every model

In [ ]:
seqs = [seq2int(r["seq"]) for r in DESIGNS]
E_ALL = {S: energies(*PARAMS[S], seqs) for S in MODELS}
for i, r in enumerate(DESIGNS):
    r["E"] = {S: E_ALL[S][i] for S in MODELS}

def designs(model, b):        return [r for r in DESIGNS if r["model"] == model and r["bin"] == b]
def E_func(scorer, gen, b):   return np.array([r["E"][scorer] for r in designs(gen, b) if r["functional"]])

## Result 1 — recognition-asymmetry / energy-penalty matrix

For an ordered pair (generator $G$, scorer $S$) at a fixed divergence, the **penalty** is
$$\mathrm{median}\big[E_S \text{ over } G\text{'s functional enzymes}\big]-\mathrm{median}\big[E_S \text{ over } S\text{'s own functional enzymes}\big].$$
A value near $0$ means the scorer regards those enzymes as just as probable as its own; a
large positive value means it treats them as far less probable. (This is threshold-free and
matched by divergence, so it is unaffected by the top-decile curation of the tested libraries.)

In [ ]:
def penalty_matrix(b):
    M = np.full((len(MODELS), len(MODELS)), np.nan)
    for gi, G in enumerate(MODELS):
        for si, S in enumerate(MODELS):
            eg, es = E_func(S, G, b), E_func(S, S, b)
            if len(eg) and len(es):
                M[gi, si] = np.median(eg) - np.median(es)
    return M

for b in ["20-25%", "40-45%"]:
    M = penalty_matrix(b)
    print(f"\n=== energy-penalty matrix, {b}  (rows = generator, cols = scorer) ===")
    print("            " + "".join(f"{S:>9}" for S in MODELS))
    for gi, G in enumerate(MODELS):
        print(f"{G:>10}  " + "".join(f"{M[gi, si]:+9.1f}" for si in range(len(MODELS))))
    for si, S in enumerate(MODELS):
        others = [M[gi, si] for gi, G in enumerate(MODELS) if G != S]
        print(f"   mean penalty imposed by {S} on the other three: {np.nanmean(others):+.1f}")

### LaTeX for Table 1 of the letter (auto-generated, to compare against the letter)

In [ ]:
for b in ["20-25%", "40-45%"]:
    M = penalty_matrix(b)
    print(f"% {b} divergence")
    for gi, G in enumerate(MODELS):
        print(f"{G} & " + " & ".join(f"${M[gi, si]:+.1f}$" for si in range(len(MODELS))) + r" \\")
    print()

### Figure — energy-penalty heatmap (`fig_penalty_matrix.pdf`)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
for ax, b in zip(axes, ["20-25%", "40-45%"]):
    M = penalty_matrix(b)
    im = ax.imshow(M, cmap="RdBu_r", vmin=-35, vmax=35, aspect="auto")
    ax.set_xticks(range(4)); ax.set_xticklabels([f"{m}\nS={ENTROPY[m]:.0f}" for m in MODELS], fontsize=8)
    ax.set_yticks(range(4)); ax.set_yticklabels(MODELS, fontsize=9)
    ax.set_xlabel("scoring model"); ax.set_ylabel("generating model"); ax.set_title(f"{b} divergence")
    for gi in range(4):
        for si in range(4):
            ax.text(si, gi, f"{M[gi, si]:+.0f}", ha="center", va="center",
                    color="white" if abs(M[gi, si]) > 18 else "black", fontsize=9)
fig.suptitle("Energy penalty a model assigns to functional enzymes it did NOT generate\n"
             "(median energy gap vs the scorer's own functional enzymes; higher = less probable)", fontsize=10)
fig.colorbar(im, ax=axes, fraction=0.025, label="energy penalty")
fig.savefig(os.path.join(OUTDIR, "fig_penalty_matrix.pdf"), bbox_inches="tight")
plt.show()

### Figure — asymmetry money-shot (`fig_asymmetry.pdf`)

Same functional enzymes, seen by the lowest- and highest-entropy models. Blue: eaDCA's functional enzymes scored by meDCA (recognised, ~0). Red: meDCA's functional enzymes scored by eaDCA (shifted into eaDCA's improbable tail).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.3))
for ax, b in zip(axes, ["20-25%", "40-45%"]):
    red  = E_func("eaDCA", "meDCA", b) - np.median(E_func("eaDCA", "eaDCA", b))
    blue = E_func("meDCA", "eaDCA", b) - np.median(E_func("meDCA", "meDCA", b))
    bb = np.linspace(min(red.min(), blue.min()) - 3, max(red.max(), blue.max()) + 3, 32)
    ax.hist(blue, bins=bb, density=True, alpha=.65, color="#37c",
            label="eaDCA's functional enzymes,\nscored by meDCA (high S)")
    ax.hist(red,  bins=bb, density=True, alpha=.65, color="#c44",
            label="meDCA's functional enzymes,\nscored by eaDCA (low S)")
    ax.axvline(0, color="k", ls="--", lw=1)
    ax.set_title(f"{b} divergence"); ax.set_ylabel("density")
    ax.set_xlabel("energy relative to scorer's own functional enzymes"); ax.legend(fontsize=8)
fig.suptitle("High-entropy model recognises the low-entropy model's functional enzymes (blue); "
             "the reverse fails (red)", fontsize=9)
fig.tight_layout()
fig.savefig(os.path.join(OUTDIR, "fig_asymmetry.pdf"), bbox_inches="tight")
plt.show()

## Result 2 — functional enzymes the low-entropy models cannot generate

For each experimentally **functional** meDCA enzyme we record its percentile under every
model (percentile of its energy within that model's own designs at the same divergence;
$100\%$ = more improbable than every design that model produced) and whether it exceeds the
**maximum** energy the model assigns to any of its own designs (`beyond_max`, i.e. that model
would essentially never generate it). Enzymes flagged `beyond_all_three` are beyond the range
of eaDCA, bmDCA **and** edDCA simultaneously. The four shown in the letter's Table 2 are the
highest-enrichment such enzymes with a *typical* meDCA percentile.

In [ ]:
def percentile(x, arr):        # % of arr strictly below x
    return 100 * np.mean(arr < x)

records = []
for b in BINS:
    ref = {S: np.array([r["E"][S] for r in designs(S, b)]) for S in MODELS}
    for r in designs("meDCA", b):
        if not r["functional"]:
            continue
        rec = {"sid": re.split(r"\s*\|\s*DCAenergy", r["name"])[0].strip(),
               "bin": b, "re": r["re"], "seq": r["seq"]}
        for S in MODELS:
            rec[S + "_pct"] = round(percentile(r["E"][S], ref[S]), 1)
            rec[S + "_beyond_max"] = bool(r["E"][S] > ref[S].max())
        rec["beyond_all_three"] = all(rec[S + "_beyond_max"] for S in ["eaDCA", "bmDCA", "edDCA"])
        records.append(rec)

star = sorted([r for r in records if r["bin"] == "20-25%" and r["beyond_all_three"]],
              key=lambda r: -r["re"])
print(f"{len(star)} functional meDCA enzymes at 20-25% are beyond ALL three other models' generated range.\n")
print(f'{"enzyme":24s} {"RE":>6} {"meDCA%":>7} {"eaDCA%":>7} {"bmDCA%":>7} {"edDCA%":>7}')
for r in star:
    print(f'{r["sid"]:24s} {r["re"]:+6.2f} {r["meDCA_pct"]:7.0f} {r["eaDCA_pct"]:7.1f} {r["bmDCA_pct"]:7.1f} {r["edDCA_pct"]:7.1f}')
    print("   " + r["seq"])

### Write the full supplementary CSV (`functional_meDCA_enzymes_full.csv`)

In [ ]:
cols = ["sid", "bin", "re", "beyond_all_three",
        "eaDCA_beyond_max", "bmDCA_beyond_max", "edDCA_beyond_max",
        "meDCA_pct", "eaDCA_pct", "bmDCA_pct", "edDCA_pct", "seq"]
out = os.path.join(OUTDIR, "functional_meDCA_enzymes_full.csv")
with open(out, "w", newline="") as fh:
    w = csv.DictWriter(fh, fieldnames=cols); w.writeheader()
    for r in sorted(records, key=lambda r: (r["bin"], -r["re"])):
        w.writerow({k: r[k] for k in cols})
n_flag = sum(r["beyond_all_three"] for r in records)
print(f"wrote {out}  ({len(records)} functional meDCA enzymes; {n_flag} flagged beyond_all_three)")

### LaTeX for Table 2 of the letter (top four enzymes)

In [ ]:
letter_four = ["meDCA batch 1 | seq 147", "meDCA batch 1 | seq 72",
               "meDCA batch 1 | seq 250", "meDCA batch 1 | seq 233"]
by_sid = {r["sid"]: r for r in records}
for sid in letter_four:
    r = by_sid[sid]
    tag = "meDCA-" + sid.split()[-1]
    print(f'{tag} & ${r["re"]:+.2f}$ & {r["meDCA_pct"]:.0f} '+r'\\')
    print(r'\multicolumn{3}{@{}l@{}}{\ttfamily\scriptsize ' + r["seq"] + r'}\\')

## Optional — threshold-free rank statistic

Probability that a *functional* enzyme from `gen` is scored **worse** (higher energy) by
`scorer` than the scorer's own functional enzymes. This is the rank companion to the energy
gaps above; note the letter quotes the **energy magnitudes**, not this rank (a small but
consistent shift can give a moderate rank probability).

In [ ]:
def p_worse(scorer, gen, b):
    eg, es = E_func(scorer, gen, b), E_func(scorer, scorer, b)
    return np.mean([x > y for x in eg for y in es]) if len(eg) and len(es) else np.nan

for b in ["20-25%", "40-45%"]:
    print(f"{b}:  eaDCA on meDCA-functional  P(worse)={p_worse('eaDCA','meDCA',b):.2f}   |   "
          f"meDCA on eaDCA-functional  P(worse)={p_worse('meDCA','eaDCA',b):.2f}")

## Scope / caveats

- The claim is about **generative coverage** (which functional enzymes a model can realistically
  produce), *not* energy as a fine predictor of activity: within the top-decile tested libraries,
  energy separates functional from non-functional only weakly (range restriction from the
  curation). The penalty matrix and `beyond_max` flags are coverage statements.
- The meDCA/HighEnt library was tested in a **separate experimental batch** from the other three
  models, with the natural library as a shared internal control; functional labels are therefore
  each experiment's own null-mode 3σ classification. Energies are computed from sequence alone and
  are unaffected by the batch.
- `beyond_max` uses each model's own tested (top-decile) designs as the reference for "would never
  generate"; because that reference is itself the model's lowest-energy sequences, it is a strong,
  conservative bar for the *other* models scoring meDCA's enzymes.